In [ ]:
import os
import re
import math
import json
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from datasets import load_dataset
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
MAX_VOCAB_SIZE = 30000
CONTEXT_LENGTH = 32
EMBEDDING_DIM = 256
LSTM_UNITS_1 = 384
LSTM_UNITS_2 = 256
DROPOUT_RATE = 0.25
RECURRENT_DROPOUT = 0.0
BATCH_SIZE = 256
EPOCHS = 20

TRAIN_TOKEN_LIMIT = 12000000
VALIDATION_TOKEN_LIMIT = 600000
TEST_TOKEN_LIMIT = 600000

MIN_TOKEN_FREQUENCY = 2
CACHE_DATA = True
USE_MIXED_PRECISION = True

if USE_MIXED_PRECISION and tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")

print({
    "MAX_VOCAB_SIZE": MAX_VOCAB_SIZE,
    "CONTEXT_LENGTH": CONTEXT_LENGTH,
    "EMBEDDING_DIM": EMBEDDING_DIM,
    "LSTM_UNITS_1": LSTM_UNITS_1,
    "LSTM_UNITS_2": LSTM_UNITS_2,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "TRAIN_TOKEN_LIMIT": TRAIN_TOKEN_LIMIT,
    "VALIDATION_TOKEN_LIMIT": VALIDATION_TOKEN_LIMIT,
    "TEST_TOKEN_LIMIT": TEST_TOKEN_LIMIT
})

In [ ]:
dataset = load_dataset("Salesforce/wikitext", "wikitext-103-v1")

print(dataset)
print(dataset["train"].features)
print(dataset["train"][0])

In [ ]:
split_sizes = {
    "train": len(dataset["train"]),
    "validation": len(dataset["validation"]),
    "test": len(dataset["test"])
}

size_df = pd.DataFrame({"split": list(split_sizes.keys()), "rows": list(split_sizes.values())})

plt.figure(figsize=(10, 6))
plt.bar(size_df["split"], size_df["rows"])
plt.title("WikiText-103 Split Sizes")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Rows")
plt.tight_layout()
plt.show()

In [ ]:
train_lines = [x for x in dataset["train"]["text"] if isinstance(x, str) and x.strip()]
validation_lines = [x for x in dataset["validation"]["text"] if isinstance(x, str) and x.strip()]
test_lines = [x for x in dataset["test"]["text"] if isinstance(x, str) and x.strip()]

print("Train lines:", len(train_lines))
print("Validation lines:", len(validation_lines))
print("Test lines:", len(test_lines))
print("Sample:")
print(train_lines[0][:1000])

In [ ]:
def clean_wikitext(text):
    text = text.replace(" @-@ ", "-")
    text = text.replace(" @.@ ", ".")
    text = text.replace(" @,@ ", ",")
    text = text.replace("= = =", " ")
    text = text.replace("= =", " ")
    text = text.replace("=", " ")
    text = text.replace("<unk>", " unk ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

train_clean_lines = [clean_wikitext(x) for x in train_lines]
validation_clean_lines = [clean_wikitext(x) for x in validation_lines]
test_clean_lines = [clean_wikitext(x) for x in test_lines]

train_clean_lines = [x for x in train_clean_lines if x]
validation_clean_lines = [x for x in validation_clean_lines if x]
test_clean_lines = [x for x in test_clean_lines if x]

print(train_clean_lines[0][:1000])

In [ ]:
line_lengths = pd.DataFrame({
    "train": pd.Series([len(x.split()) for x in train_clean_lines]),
    "validation": pd.Series([len(x.split()) for x in validation_clean_lines]),
    "test": pd.Series([len(x.split()) for x in test_clean_lines])
})

plt.figure(figsize=(11, 6))
plt.hist(line_lengths["train"].dropna(), bins=80, alpha=0.75)
plt.title("Distribution of Train Line Lengths")
plt.xlabel("Words per Line")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

print(line_lengths.describe())

In [ ]:
train_text = "\n".join(train_clean_lines)
validation_text = "\n".join(validation_clean_lines)
test_text = "\n".join(test_clean_lines)

train_word_list = re.findall(r"\S+", train_text)
validation_word_list = re.findall(r"\S+", validation_text)
test_word_list = re.findall(r"\S+", test_text)

print("Train tokens available:", len(train_word_list))
print("Validation tokens available:", len(validation_word_list))
print("Test tokens available:", len(test_word_list))

In [ ]:
def limit_tokens(tokens, limit):
    if limit is None or len(tokens) <= limit:
        return tokens
    return tokens[:limit]

train_tokens = limit_tokens(train_word_list, TRAIN_TOKEN_LIMIT)
validation_tokens = limit_tokens(validation_word_list, VALIDATION_TOKEN_LIMIT)
test_tokens = limit_tokens(test_word_list, TEST_TOKEN_LIMIT)

print("Training tokens used:", len(train_tokens))
print("Validation tokens used:", len(validation_tokens))
print("Test tokens used:", len(test_tokens))

In [ ]:
from collections import Counter

frequency = Counter(train_tokens)

freq_df = pd.DataFrame(
    frequency.most_common(30),
    columns=["word", "frequency"]
)

plt.figure(figsize=(12, 7))
plt.barh(freq_df["word"][::-1], freq_df["frequency"][::-1])
plt.title("Top 30 Words in the Training Corpus")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.show()

print(freq_df)

In [ ]:
vocab_counts = Counter(train_tokens)
kept_words = {w for w, c in vocab_counts.items() if c >= MIN_TOKEN_FREQUENCY}

word_to_id = {"<PAD>": 0, "<UNK>": 1}

for word, count in vocab_counts.most_common():
    if word in kept_words:
        if len(word_to_id) >= MAX_VOCAB_SIZE:
            break
        word_to_id[word] = len(word_to_id)

id_to_word = {idx: word for word, idx in word_to_id.items()}

print("Vocabulary size:", len(word_to_id))
print("Top vocabulary:", list(word_to_id.items())[:25])

In [ ]:
def encode_tokens(tokens, mapping):
    unk_id = mapping["<UNK>"]
    return np.asarray([mapping.get(token, unk_id) for token in tokens], dtype=np.int32)

train_ids = encode_tokens(train_tokens, word_to_id)
validation_ids = encode_tokens(validation_tokens, word_to_id)
test_ids = encode_tokens(test_tokens, word_to_id)

print("Encoded train shape:", train_ids.shape)
print("Encoded validation shape:", validation_ids.shape)
print("Encoded test shape:", test_ids.shape)
print("UNK ratio train:", float(np.mean(train_ids == 1)))
print("UNK ratio validation:", float(np.mean(validation_ids == 1)))
print("UNK ratio test:", float(np.mean(test_ids == 1)))

In [ ]:
def show_encoded_window(tokens, ids, start=0, width=25):
    sample_tokens = tokens[start:start + width]
    sample_ids = ids[start:start + width]
    print(pd.DataFrame({"token": sample_tokens, "id": sample_ids}))

show_encoded_window(train_tokens, train_ids)

In [ ]:
def count_examples(num_tokens, context_length):
    return max(0, num_tokens - context_length)

example_counts = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "tokens": [len(train_ids), len(validation_ids), len(test_ids)],
    "examples": [
        count_examples(len(train_ids), CONTEXT_LENGTH),
        count_examples(len(validation_ids), CONTEXT_LENGTH),
        count_examples(len(test_ids), CONTEXT_LENGTH)
    ]
})

display(example_counts)

In [ ]:
def make_windows(token_ids, context_length):
    x = token_ids[:-1]
    y = token_ids[context_length:]
    x = tf.keras.utils.timeseries_dataset_from_array(
        x,
        targets=y,
        sequence_length=context_length,
        sequence_stride=1,
        sampling_rate=1,
        batch_size=None,
        shuffle=False
    )
    return x

train_ds_raw = make_windows(train_ids, CONTEXT_LENGTH)
validation_ds_raw = make_windows(validation_ids, CONTEXT_LENGTH)
test_ds_raw = make_windows(test_ids, CONTEXT_LENGTH)

sample_x, sample_y = next(iter(train_ds_raw))
print("Input shape:", sample_x.shape)
print("Target:", sample_y.numpy())
print("Decoded input:", [id_to_word.get(int(i), "<UNK>") for i in sample_x.numpy()])
print("Decoded target:", id_to_word.get(int(sample_y.numpy()), "<UNK>"))

In [ ]:
train_ds = train_ds_raw.shuffle(100000, seed=SEED, reshuffle_each_iteration=True).batch(BATCH_SIZE)
validation_ds = validation_ds_raw.batch(BATCH_SIZE)
test_ds = test_ds_raw.batch(BATCH_SIZE)

if CACHE_DATA:
    train_ds = train_ds.cache().prefetch(AUTOTUNE)
    validation_ds = validation_ds.cache().prefetch(AUTOTUNE)
    test_ds = test_ds.cache().prefetch(AUTOTUNE)
else:
    train_ds = train_ds.prefetch(AUTOTUNE)
    validation_ds = validation_ds.prefetch(AUTOTUNE)
    test_ds = test_ds.prefetch(AUTOTUNE)

print("Datasets ready")

In [ ]:
def decode_sequence(ids):
    return " ".join(id_to_word.get(int(i), "<UNK>") for i in ids)

for xb, yb in train_ds.take(1):
    for i in range(3):
        print("Input:", decode_sequence(xb[i].numpy()))
        print("Target:", id_to_word.get(int(yb[i].numpy()), "<UNK>"))
        print()

In [ ]:
inputs = keras.Input(shape=(CONTEXT_LENGTH,), dtype=tf.int32)

x = layers.Embedding(
    input_dim=len(word_to_id),
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="embedding"
)(inputs)

x = layers.LayerNormalization(name="embedding_norm")(x)

x = layers.LSTM(
    LSTM_UNITS_1,
    return_sequences=True,
    dropout=DROPOUT_RATE,
    recurrent_dropout=RECURRENT_DROPOUT,
    name="lstm_1"
)(x)

x = layers.LayerNormalization(name="lstm_1_norm")(x)

x = layers.LSTM(
    LSTM_UNITS_2,
    return_sequences=False,
    dropout=DROPOUT_RATE,
    recurrent_dropout=RECURRENT_DROPOUT,
    name="lstm_2"
)(x)

x = layers.LayerNormalization(name="lstm_2_norm")(x)
x = layers.Dropout(DROPOUT_RATE, name="final_dropout")(x)

outputs = layers.Dense(
    len(word_to_id),
    dtype="float32",
    name="next_word"
)(x)

model = keras.Model(inputs, outputs)

model.summary()

In [ ]:
trainable_params = np.sum([np.prod(v.shape) for v in model.trainable_weights])
non_trainable_params = np.sum([np.prod(v.shape) for v in model.non_trainable_weights])

parameter_df = pd.DataFrame({
    "type": ["Trainable", "Non-trainable", "Total"],
    "parameters": [
        trainable_params,
        non_trainable_params,
        trainable_params + non_trainable_params
    ]
})

display(parameter_df)

In [ ]:
try:
    keras.utils.plot_model(
        model,
        to_file="wikitext_lstm_architecture.png",
        show_shapes=True,
        show_layer_names=True,
        expand_nested=True,
        dpi=140
    )
    display(keras.utils.load_img("wikitext_lstm_architecture.png"))
except Exception as e:
    print(e)

In [ ]:
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

optimizer = keras.optimizers.AdamW(
    learning_rate=2e-3,
    weight_decay=1e-4,
    clipnorm=1.0
)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top_3_accuracy"),
        keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy")
    ]
)

print(model.optimizer.get_config())

In [ ]:
os.makedirs("checkpoints", exist_ok=True)

callbacks = [
    ModelCheckpoint(
        "checkpoints/best_wikitext_lstm.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    CSVLogger("training_history.csv")
]

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(history_df["loss"], label="Train Loss")
ax.plot(history_df["val_loss"], label="Validation Loss")
ax.set_title("Training and Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(history_df["accuracy"], label="Train Top-1 Accuracy")
ax.plot(history_df["val_accuracy"], label="Validation Top-1 Accuracy")
ax.plot(history_df["top_3_accuracy"], label="Train Top-3 Accuracy")
ax.plot(history_df["val_top_3_accuracy"], label="Validation Top-3 Accuracy")
ax.set_title("Top-1 and Top-3 Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(history_df["top_5_accuracy"], label="Train Top-5 Accuracy")
ax.plot(history_df["val_top_5_accuracy"], label="Validation Top-5 Accuracy")
ax.set_title("Top-5 Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(history_df["loss"], label="Train Loss")
ax.plot(history_df["val_loss"], label="Validation Loss")
ax.set_yscale("log")
ax.set_title("Loss on Log Scale")
ax.set_xlabel("Epoch")
ax.set_ylabel("Log Loss")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_epoch = int(np.argmin(history_df["val_loss"].values)) + 1
best_val_loss = float(history_df["val_loss"].min())
best_val_accuracy = float(history_df.loc[history_df["val_loss"].idxmin(), "val_accuracy"])
best_val_top3 = float(history_df.loc[history_df["val_loss"].idxmin(), "val_top_3_accuracy"])
best_val_top5 = float(history_df.loc[history_df["val_loss"].idxmin(), "val_top_5_accuracy"])

summary = pd.DataFrame({
    "metric": ["Best Epoch", "Best Validation Loss", "Validation Accuracy", "Validation Top-3", "Validation Top-5"],
    "value": [best_epoch, best_val_loss, best_val_accuracy, best_val_top3, best_val_top5]
})

display(summary)

In [ ]:
test_results = model.evaluate(test_ds, return_dict=True, verbose=1)
test_results_df = pd.DataFrame({
    "metric": list(test_results.keys()),
    "value": list(test_results.values())
})
display(test_results_df)

In [ ]:
test_perplexity = math.exp(float(test_results["loss"]))
val_perplexity = math.exp(float(best_val_loss))

perplexity_df = pd.DataFrame({
    "split": ["validation", "test"],
    "perplexity": [val_perplexity, test_perplexity]
})

display(perplexity_df)

plt.figure(figsize=(9, 6))
plt.bar(perplexity_df["split"], perplexity_df["perplexity"])
plt.title("Language Model Perplexity")
plt.xlabel("Split")
plt.ylabel("Perplexity")
plt.tight_layout()
plt.show()

In [ ]:
def encode_prompt(prompt):
    tokens = re.findall(r"\S+", clean_wikitext(prompt))
    ids = [word_to_id.get(token, word_to_id["<UNK>"]) for token in tokens]
    if len(ids) < CONTEXT_LENGTH:
        ids = [word_to_id["<PAD>"]] * (CONTEXT_LENGTH - len(ids)) + ids
    else:
        ids = ids[-CONTEXT_LENGTH:]
    return np.asarray([ids], dtype=np.int32)

def predict_next_word(prompt, top_k=10, temperature=1.0):
    x = encode_prompt(prompt)
    logits = model.predict(x, verbose=0)[0]
    logits = logits / max(float(temperature), 1e-6)
    values, indices = tf.math.top_k(logits, k=top_k)
    probabilities = tf.nn.softmax(values).numpy()
    words = [id_to_word.get(int(i), "<UNK>") for i in indices.numpy()]
    return pd.DataFrame({
        "word": words,
        "probability": probabilities
    })

seed_prompts = [
    "the history of",
    "in the middle of",
    "one of the most",
    "the first time"
]

for prompt in seed_prompts:
    print("Prompt:", prompt)
    display(predict_next_word(prompt, top_k=8, temperature=0.9))

In [ ]:
prompt = "the history of"
prediction_df = predict_next_word(prompt, top_k=12, temperature=0.8)

plt.figure(figsize=(12, 7))
plt.barh(prediction_df["word"][::-1], prediction_df["probability"][::-1])
plt.title(f"Next-Word Distribution: {prompt}")
plt.xlabel("Probability")
plt.ylabel("Candidate Word")
plt.tight_layout()
plt.show()

In [ ]:
def sample_from_logits(logits, temperature=0.8, top_k=10):
    logits = np.asarray(logits, dtype=np.float64) / max(float(temperature), 1e-6)
    if top_k is not None and top_k < len(logits):
        indices = np.argpartition(logits, -top_k)[-top_k:]
        filtered = np.full_like(logits, -np.inf)
        filtered[indices] = logits[indices]
        logits = filtered
    probabilities = tf.nn.softmax(logits).numpy()
    return int(np.random.choice(len(probabilities), p=probabilities))

def generate_text(seed_text, num_words=40, temperature=0.8, top_k=10):
    generated = clean_wikitext(seed_text)
    for _ in range(num_words):
        x = encode_prompt(generated)
        logits = model.predict(x, verbose=0)[0]
        next_id = sample_from_logits(logits, temperature=temperature, top_k=top_k)
        next_word = id_to_word.get(next_id, "<UNK>")
        generated = generated + " " + next_word
    return generated

generation_settings = [
    (0.45, 5),
    (0.70, 10),
    (0.90, 20),
    (1.10, 40)
]

seed = "the future of science"

for temperature, top_k in generation_settings:
    print("Temperature:", temperature, "| Top-K:", top_k)
    print(generate_text(seed, num_words=35, temperature=temperature, top_k=top_k))
    print()

In [ ]:
generation_lengths = [15, 30, 45, 60, 90]

generated_samples = []

for length in generation_lengths:
    generated = generate_text(
        "the world of technology",
        num_words=length,
        temperature=0.75,
        top_k=15
    )
    generated_samples.append({
        "requested_words": length,
        "generated_words": len(generated.split()),
        "text": generated
    })

generation_df = pd.DataFrame(generated_samples)
display(generation_df)

In [ ]:
sample_prompts = [
    "the government of",
    "the development of",
    "the people of",
    "the importance of",
    "the end of",
    "the beginning of",
    "one of the",
    "according to the"
]

rows = []

for prompt in sample_prompts:
    predictions = predict_next_word(prompt, top_k=1, temperature=0.8)
    predicted = predictions.iloc[0]["word"]
    probability = float(predictions.iloc[0]["probability"])
    rows.append([prompt, predicted, probability])

prediction_eval_df = pd.DataFrame(
    rows,
    columns=["prompt", "predicted_next_word", "confidence"]
)

display(prediction_eval_df)

plt.figure(figsize=(12, 7))
plt.barh(
    prediction_eval_df["prompt"][::-1],
    prediction_eval_df["confidence"][::-1]
)
plt.title("Model Confidence for Sample Prompts")
plt.xlabel("Top-1 Probability")
plt.ylabel("Prompt")
plt.tight_layout()
plt.show()

In [ ]:
train_frequency = np.bincount(train_ids, minlength=len(word_to_id))
validation_frequency = np.bincount(validation_ids, minlength=len(word_to_id))
test_frequency = np.bincount(test_ids, minlength=len(word_to_id))

frequency_comparison = pd.DataFrame({
    "id": np.arange(len(word_to_id)),
    "word": [id_to_word.get(i, "<UNK>") for i in range(len(word_to_id))],
    "train": train_frequency,
    "validation": validation_frequency,
    "test": test_frequency
})

frequency_comparison["total"] = (
    frequency_comparison["train"]
    + frequency_comparison["validation"]
    + frequency_comparison["test"]
)

display(frequency_comparison.sort_values("total", ascending=False).head(25))

In [ ]:
word_lengths = np.asarray([len(word) for word in word_to_id.keys()])

plt.figure(figsize=(10, 6))
plt.hist(word_lengths, bins=30)
plt.title("Vocabulary Word-Length Distribution")
plt.xlabel("Characters per Word")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Mean word length:", word_lengths.mean())
print("Median word length:", np.median(word_lengths))

In [ ]:
embedding_weights = model.get_layer("embedding").get_weights()[0]

embedding_norms = np.linalg.norm(embedding_weights, axis=1)

plt.figure(figsize=(10, 6))
plt.hist(embedding_norms, bins=60)
plt.title("Embedding Vector Norm Distribution")
plt.xlabel("L2 Norm")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "metric": ["Validation Accuracy", "Validation Top-3", "Validation Top-5", "Validation Perplexity", "Test Accuracy", "Test Top-3", "Test Top-5", "Test Perplexity"],
    "value": [
        best_val_accuracy,
        best_val_top3,
        best_val_top5,
        val_perplexity,
        test_results["accuracy"],
        test_results["top_3_accuracy"],
        test_results["top_5_accuracy"],
        test_perplexity
    ]
})

display(comparison)

plt.figure(figsize=(12, 6))
plt.bar(comparison["metric"], comparison["value"])
plt.title("Final Model Evaluation")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
with open("wikitext103_word_to_id.json", "w", encoding="utf-8") as f:
    json.dump(word_to_id, f, ensure_ascii=False)

with open("wikitext103_id_to_word.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in id_to_word.items()}, f, ensure_ascii=False)

model.save("wikitext103_lstm_next_word_predictor.keras")

print("Saved:")
print("wikitext103_lstm_next_word_predictor.keras")
print("wikitext103_word_to_id.json")
print("wikitext103_id_to_word.json")
print("training_history.csv")

In [ ]:
final_report = {
    "dataset": "WikiText-103 v1",
    "vocabulary_size": len(word_to_id),
    "context_length": CONTEXT_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "lstm_units": [LSTM_UNITS_1, LSTM_UNITS_2],
    "training_tokens": len(train_ids),
    "validation_tokens": len(validation_ids),
    "test_tokens": len(test_ids),
    "trainable_parameters": int(trainable_params),
    "best_epoch": best_epoch,
    "validation_accuracy": float(best_val_accuracy),
    "validation_top_3_accuracy": float(best_val_top3),
    "validation_top_5_accuracy": float(best_val_top5),
    "validation_perplexity": float(val_perplexity),
    "test_accuracy": float(test_results["accuracy"]),
    "test_top_3_accuracy": float(test_results["top_3_accuracy"]),
    "test_top_5_accuracy": float(test_results["top_5_accuracy"]),
    "test_perplexity": float(test_perplexity)
}

print(json.dumps(final_report, indent=2))